In [ ]:
# Cell 1: GPU Check
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'GPU Available: {len(gpus) > 0}')
if len(gpus) == 0:
    print('No GPU! Go to: Runtime → Change runtime type → GPU')
else:
    print('GPU ready!')

In [ ]:
# Cell 2: Clone GitHub Repo
import os
GITHUB_USERNAME = 'Aditya-Hippargi'
REPO_NAME = 'Diabetic-Retinopathy'
GITHUB_TOKEN = 'YOUR_PERSONAL_ACCESS_TOKEN'  # paste your token here

if os.path.exists(f'/content/{REPO_NAME}'):
    !rm -rf /content/{REPO_NAME}

!git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git
%cd /content/{REPO_NAME}
!ls -la

In [ ]:
# Cell 3: Install Libraries
!pip install -q opencv-python-headless scikit-learn
print('Libraries installed!')

In [ ]:
# Cell 4: Upload trained model files
# Upload efficientnet_best.keras and resnet_best.keras from your laptop
from google.colab import files
import os

os.makedirs('models', exist_ok=True)
print('Upload your trained model files:')
print('  1. efficientnet_best.keras')
print('  2. resnet_best.keras')
uploaded = files.upload()

# Move uploaded files to models folder
for filename in uploaded.keys():
    os.rename(filename, f'models/{filename}')
    print(f'Moved {filename} to models/')

print('\nFiles in models folder:')
!ls -la models/

In [ ]:
# Cell 5: Download Preprocessed Images from Kaggle
print('Upload your kaggle.json file:')
uploaded = files.upload()
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!mkdir -p data/processed
!kaggle datasets download -d jeetpaghdar/aptos-preprocessed-train -p data/processed/
!unzip -q data/processed/aptos-preprocessed-train.zip -d data/processed/train_images/
!rm data/processed/aptos-preprocessed-train.zip
count = len(os.listdir('data/processed/train_images/'))
print(f'Done! {count} images ready')

In [ ]:
# Cell 6: Download CSV Labels
import pandas as pd
!mkdir -p data/raw
!kaggle competitions download -c aptos2019-blindness-detection -p data/raw/ --file train.csv
!unzip -q data/raw/train.csv.zip -d data/raw/ 2>/dev/null || true
train_csv = pd.read_csv('data/raw/train.csv')
print(f'Loaded {len(train_csv)} labels')
print(train_csv['diagnosis'].value_counts().sort_index())

In [ ]:
# Cell 7: Import Libraries
import sys, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import cv2, os, json, warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score,
    classification_report, confusion_matrix
)

np.random.seed(42)
tf.random.set_seed(42)
print(f'Imports done | TF: {tf.__version__}')

In [ ]:
# Cell 8: Prepare validation dataset
train_csv = pd.read_csv('data/raw/train.csv')
train_csv['image_path'] = train_csv['id_code'].apply(
    lambda x: f'data/processed/train_images/{x}.png'
)
train_csv = train_csv[train_csv['image_path'].apply(os.path.exists)].reset_index(drop=True)

# Same split as training — random_state=42 ensures same val set
train_df, val_df = train_test_split(
    train_csv, test_size=0.20,
    stratify=train_csv['diagnosis'], random_state=42
)
val_df = val_df.reset_index(drop=True)

print(f'Validation samples: {len(val_df)}')
print(val_df['diagnosis'].value_counts().sort_index())

In [ ]:
# Cell 9: Create validation tf.data dataset
# Use 224x224 for both models (common size)
IMG_SIZE   = 224
BATCH_SIZE = 32

def load_image(image_path, label):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

val_paths  = val_df['image_path'].values
val_labels = tf.keras.utils.to_categorical(val_df['diagnosis'].values, num_classes=5)

val_dataset = (tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
               .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
               .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

print(f'Val batches: {len(val_dataset)}')

In [ ]:
# Cell 10: Load both trained models
efficientnet_model = keras.models.load_model('models/efficientnet_best.keras')
resnet_model       = keras.models.load_model('models/resnet_best.keras')

print('Both models loaded!')
print(f'EfficientNet input shape: {efficientnet_model.input_shape}')
print(f'ResNet50 input shape    : {resnet_model.input_shape}')

In [ ]:
# Cell 11: Get predictions from both models individually
# We collect raw probabilities (softmax output) not just class labels
# This allows us to average probabilities in the ensemble

print('Getting EfficientNet predictions...')
eff_probs = []
y_true    = []

for X_batch, y_batch in val_dataset:
    # Resize to 224 for EfficientNet
    X_224 = tf.image.resize(X_batch, [224, 224])
    probs = efficientnet_model.predict(X_224, verbose=0)
    eff_probs.extend(probs)
    y_true.extend(np.argmax(y_batch.numpy(), axis=1))

print('Getting ResNet50 predictions...')
res_probs = []

for X_batch, y_batch in val_dataset:
    # Resize to 224 for ResNet
    X_224 = tf.image.resize(X_batch, [224, 224])
    probs = resnet_model.predict(X_224, verbose=0)
    res_probs.extend(probs)

eff_probs = np.array(eff_probs)
res_probs = np.array(res_probs)
y_true    = np.array(y_true)

print(f'EfficientNet probs shape: {eff_probs.shape}')
print(f'ResNet50 probs shape    : {res_probs.shape}')
print(f'True labels shape       : {y_true.shape}')

In [ ]:
# Cell 12: Individual model results
# Evaluate each model separately before combining
eff_preds = np.argmax(eff_probs, axis=1)
res_preds = np.argmax(res_probs, axis=1)

eff_acc   = accuracy_score(y_true, eff_preds)
res_acc   = accuracy_score(y_true, res_preds)
eff_kappa = cohen_kappa_score(y_true, eff_preds, weights='quadratic')
res_kappa = cohen_kappa_score(y_true, res_preds, weights='quadratic')

print('='*60)
print('  INDIVIDUAL MODEL RESULTS')
print('='*60)
print(f'  EfficientNetB3 : Accuracy={eff_acc*100:.2f}% | Kappa={eff_kappa:.4f}')
print(f'  ResNet50       : Accuracy={res_acc*100:.2f}% | Kappa={res_kappa:.4f}')
print('='*60)

In [ ]:
# Cell 13: Ensemble — try different weight combinations
# We test multiple weight ratios to find the best combination
# Weight ratio = how much to trust each model's prediction

print('Testing different ensemble weights...')
print('='*60)

best_acc   = 0
best_kappa = 0
best_w_acc = None
best_w_kappa = None

# Test weights from 0.1 to 0.9 for EfficientNet
# ResNet weight = 1 - EfficientNet weight
for w_eff in [0.3, 0.4, 0.5, 0.6, 0.7]:
    w_res = 1.0 - w_eff
    ensemble_probs = (w_eff * eff_probs) + (w_res * res_probs)
    ensemble_preds = np.argmax(ensemble_probs, axis=1)

    acc   = accuracy_score(y_true, ensemble_preds)
    kappa = cohen_kappa_score(y_true, ensemble_preds, weights='quadratic')

    print(f'  EfficientNet={w_eff:.1f} | ResNet={w_res:.1f} | Acc={acc*100:.2f}% | Kappa={kappa:.4f}')

    if acc > best_acc:
        best_acc   = acc
        best_w_acc = (w_eff, w_res)
    if kappa > best_kappa:
        best_kappa   = kappa
        best_w_kappa = (w_eff, w_res)

print('='*60)
print(f'  Best Accuracy weights : EfficientNet={best_w_acc[0]} | ResNet={best_w_acc[1]}')
print(f'  Best Kappa weights    : EfficientNet={best_w_kappa[0]} | ResNet={best_w_kappa[1]}')

In [ ]:
# Cell 14: Final ensemble with best weights
# Use the weights that gave best accuracy
w_eff = best_w_acc[0]
w_res = best_w_acc[1]

ensemble_probs = (w_eff * eff_probs) + (w_res * res_probs)
ensemble_preds = np.argmax(ensemble_probs, axis=1)

accuracy = accuracy_score(y_true, ensemble_preds)
f1       = f1_score(y_true, ensemble_preds, average='weighted')
kappa    = cohen_kappa_score(y_true, ensemble_preds, weights='quadratic')

print('='*60)
print('  ENSEMBLE FINAL RESULTS')
print('='*60)
print(f'  Weights  : EfficientNet={w_eff} | ResNet={w_res}')
print(f'  Accuracy : {accuracy*100:.2f}%')
print(f'  F1 Score : {f1:.4f}')
print(f'  Kappa    : {kappa:.4f}')
print('='*60)
print(classification_report(y_true, ensemble_preds, target_names=['No DR','Mild','Moderate','Severe','Prolif.']))

# Save results
with open('models/ensemble_results.json', 'w') as f:
    json.dump({
        'accuracy': float(accuracy),
        'f1': float(f1),
        'kappa': float(kappa),
        'efficientnet_weight': w_eff,
        'resnet_weight': w_res
    }, f, indent=2)
print('Results saved!')

In [ ]:
# Cell 15: Ensemble confusion matrix
cm = confusion_matrix(y_true, ensemble_preds)
grade_names = ['No DR', 'Mild', 'Moderate', 'Severe', 'Prolif.']

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=grade_names, yticklabels=grade_names)
plt.title(f'Ensemble — Confusion Matrix ({accuracy*100:.1f}%)', fontsize=14, fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('models/ensemble_confusion_matrix.png', dpi=300)
plt.show()
print('Confusion matrix saved!')

In [ ]:
# Cell 16: Model comparison bar chart
models      = ['Baseline CNN', 'EfficientNetB3', 'ResNet50', 'Ensemble']
accuracies  = [50.0, eff_acc*100, res_acc*100, accuracy*100]
kappas      = [0.35, eff_kappa, res_kappa, kappa]
colors      = ['#gray', '#2196F3', '#4CAF50', '#9C27B0']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Model Comparison — RetinaScan AI', fontsize=14, fontweight='bold')

# Accuracy chart
bars = axes[0].bar(models, accuracies, color=['#78909C', '#2196F3', '#4CAF50', '#9C27B0'])
axes[0].set_title('Accuracy (%)')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_ylim(0, 100)
axes[0].axhline(y=80, color='red', linestyle='--', alpha=0.5, label='80% target')
axes[0].legend()
for bar, val in zip(bars, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val:.1f}%', ha='center', fontweight='bold')

# Kappa chart
bars2 = axes[1].bar(models, kappas, color=['#78909C', '#2196F3', '#4CAF50', '#9C27B0'])
axes[1].set_title('Quadratic Weighted Kappa')
axes[1].set_ylabel('Kappa Score')
axes[1].set_ylim(0, 1)
axes[1].axhline(y=0.8, color='red', linestyle='--', alpha=0.5, label='0.8 target')
axes[1].legend()
for bar, val in zip(bars2, kappas):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('models/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('Comparison chart saved!')

In [ ]:
# Cell 17: Download all ensemble results
from google.colab import files

print('Downloading files...')
files.download('models/ensemble_confusion_matrix.png')
files.download('models/model_comparison.png')
files.download('models/ensemble_results.json')

print('\n' + '='*60)
print('  ENSEMBLE COMPLETE!')
print('='*60)
print(f'  EfficientNetB3 : {eff_acc*100:.2f}%')
print(f'  ResNet50       : {res_acc*100:.2f}%')
print(f'  Ensemble       : {accuracy*100:.2f}%')
print(f'  Kappa          : {kappa:.4f}')
print('\n  Next: Run 07_gradcam_visualization.ipynb')
print('='*60)